# Tests de utils_pytorch.py

- creation: *13/02/2025*

In [1]:
from random import randint, seed
from functools import reduce
import torch
import utils.utils_pytorch as utils_pytorch

In [2]:
seed(7)

## Outils de génération des données

In [ ]:
batch = 3
channel = 3
heigh = 5
width = 10

bc_value = lambda b, c : b * 10 + c
bc_max_value = lambda b, c, cst : bc_value(b, c) * 10 + 7 + cst

def generate_max_and_pos(batch, channel, heigh, width, gen_max=None, one_by=None):
    """
    Retourne tenseur de maximum calculés selon gen_max et leur position aléatoire 3D (batch = 0) ou 4D (batch != 0) par heigh.

    one_by in [None, "tensor", "batch", "channel"]
    """
    pos = []
    max_values = []

    range_b = [randint(0, max(1, batch)-1)] if one_by in ["tensor"] else range(max(1, batch))
    for b in range_b:
        range_c = [randint(0, channel-1)] if one_by in ["tensor", "batch"] else range(channel)
        for c in range_c:
            range_h = [randint(0, heigh-1)] if one_by in ["tensor", "batch", "channel"] else range(heigh)
            for h in range_h:
                pos.append((b, c, h, randint(0, width-1)))
                max_values.append(gen_max(b, c, h) if callable(gen_max) else gen_max)

    pos = torch.tensor(pos)
    max_values = torch.tensor(max_values)

    if batch == 0:
        pos = pos[:, 1:]

    return max_values, pos


def generate_tensor(batch, channel, heigh, width, bc_value, max_values=None, pos=None, dtype=int):
    """
    Retourne un tenseur 3D (batch = 0) ou 4D (batch > 0) remplie selon bc_value, et contenant les max_values à 
    leur position respective indiquée dans pos si fourni.
    """
    assert (max_values!= None and pos != None) or (max_values== None and pos == None), \
        "max_values and pos must be booth not rovided or provided."
    
    t = torch.zeros((max(batch, 1), channel, heigh, width), dtype=dtype)
    for b in range(max(batch, 1)):
        for c in range(channel):
            t[b, c, :, :] = bc_value(b, c) if callable(bc_value) else bc_value

    if b == 0:
        t = t[0]
    
    if max_values != None and pos != None:
        t[*pos.T] = max_values.type(t.dtype)
    
    return t

In [ ]:
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value)
print("> max_values : \n", max_values)
print("> pos :", max_pos, sep="\n")

t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)
print("t :", t, sep="\n")

> max_values : 
 tensor([  7,   8,   9,  10,  11,  17,  18,  19,  20,  21,  27,  28,  29,  30,
         31, 107, 108, 109, 110, 111, 117, 118, 119, 120, 121, 127, 128, 129,
        130, 131, 207, 208, 209, 210, 211, 217, 218, 219, 220, 221, 227, 228,
        229, 230, 231])
> pos :
tensor([[0, 0, 0, 5],
        [0, 0, 1, 2],
        [0, 0, 2, 6],
        [0, 0, 3, 0],
        [0, 0, 4, 1],
        [0, 1, 0, 8],
        [0, 1, 1, 1],
        [0, 1, 2, 5],
        [0, 1, 3, 9],
        [0, 1, 4, 0],
        [0, 2, 0, 8],
        [0, 2, 1, 3],
        [0, 2, 2, 0],
        [0, 2, 3, 1],
        [0, 2, 4, 6],
        [1, 0, 0, 6],
        [1, 0, 1, 1],
        [1, 0, 2, 3],
        [1, 0, 3, 1],
        [1, 0, 4, 8],
        [1, 1, 0, 6],
        [1, 1, 1, 0],
        [1, 1, 2, 9],
        [1, 1, 3, 1],
        [1, 1, 4, 3],
        [1, 2, 0, 9],
        [1, 2, 1, 0],
        [1, 2, 2, 9],
        [1, 2, 3, 9],
        [1, 2, 4, 6],
        [2, 0, 0, 0],
        [2, 0, 1, 3],
        [2, 0

## Tests

### `get_all_occurence_indices()`

In [ ]:
gen_max = 100
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=gen_max)
t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)

expected = max_pos
result = utils_pytorch.get_all_occurence_indices(t, gen_max)
print("> result :", result, sep="\n")
assert torch.equal(result, expected), "Error"

print("OK !!!")

> result :
tensor([[0, 0, 0, 9],
        [0, 0, 1, 9],
        [0, 0, 2, 3],
        [0, 0, 3, 5],
        [0, 0, 4, 1],
        [0, 1, 0, 8],
        [0, 1, 1, 1],
        [0, 1, 2, 9],
        [0, 1, 3, 0],
        [0, 1, 4, 9],
        [0, 2, 0, 3],
        [0, 2, 1, 7],
        [0, 2, 2, 8],
        [0, 2, 3, 6],
        [0, 2, 4, 5],
        [1, 0, 0, 7],
        [1, 0, 1, 9],
        [1, 0, 2, 7],
        [1, 0, 3, 5],
        [1, 0, 4, 4],
        [1, 1, 0, 3],
        [1, 1, 1, 2],
        [1, 1, 2, 3],
        [1, 1, 3, 1],
        [1, 1, 4, 9],
        [1, 2, 0, 4],
        [1, 2, 1, 8],
        [1, 2, 2, 7],
        [1, 2, 3, 5],
        [1, 2, 4, 7],
        [2, 0, 0, 4],
        [2, 0, 1, 9],
        [2, 0, 2, 1],
        [2, 0, 3, 1],
        [2, 0, 4, 8],
        [2, 1, 0, 6],
        [2, 1, 1, 2],
        [2, 1, 2, 5],
        [2, 1, 3, 2],
        [2, 1, 4, 7],
        [2, 2, 0, 6],
        [2, 2, 1, 0],
        [2, 2, 2, 1],
        [2, 2, 3, 8],
        [2, 2, 4, 9]]

### `get_all_max_indices()`

In [ ]:
gen_max = 1e6
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=gen_max)
t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)

expected_pos = max_pos
result_max_vector, result_pos = utils_pytorch.get_all_max_indices(t)
print("> result_max_value :", result_max_vector)
assert result_max_vector == gen_max, "Error on result_max_value"
print("> result_pos :", result_pos, sep="\n")
assert torch.equal(expected_pos, result_pos), "Error on result_pos"

print("OK !!!")

> result_max_value : 1000000
> result_pos :
tensor([[0, 0, 0, 5],
        [0, 0, 1, 5],
        [0, 0, 2, 5],
        [0, 0, 3, 9],
        [0, 0, 4, 7],
        [0, 1, 0, 9],
        [0, 1, 1, 7],
        [0, 1, 2, 1],
        [0, 1, 3, 1],
        [0, 1, 4, 4],
        [0, 2, 0, 7],
        [0, 2, 1, 1],
        [0, 2, 2, 0],
        [0, 2, 3, 4],
        [0, 2, 4, 9],
        [1, 0, 0, 7],
        [1, 0, 1, 4],
        [1, 0, 2, 6],
        [1, 0, 3, 5],
        [1, 0, 4, 0],
        [1, 1, 0, 7],
        [1, 1, 1, 5],
        [1, 1, 2, 2],
        [1, 1, 3, 9],
        [1, 1, 4, 1],
        [1, 2, 0, 7],
        [1, 2, 1, 0],
        [1, 2, 2, 3],
        [1, 2, 3, 4],
        [1, 2, 4, 2],
        [2, 0, 0, 3],
        [2, 0, 1, 6],
        [2, 0, 2, 6],
        [2, 0, 3, 7],
        [2, 0, 4, 1],
        [2, 1, 0, 2],
        [2, 1, 1, 7],
        [2, 1, 2, 6],
        [2, 1, 3, 8],
        [2, 1, 4, 4],
        [2, 2, 0, 2],
        [2, 2, 1, 6],
        [2, 2, 2, 8],
        [2

### `get_max_by_dim()`

In [ ]:
max_values, max_pos = generate_max_and_pos(batch, channel, heigh, width, gen_max=bc_max_value)
t = generate_tensor(batch, channel, heigh, width, bc_value, max_values=max_values, pos=max_pos)

In [8]:
print(max_values.size(), max_values)

torch.Size([45]) tensor([  7,   8,   9,  10,  11,  17,  18,  19,  20,  21,  27,  28,  29,  30,
         31, 107, 108, 109, 110, 111, 117, 118, 119, 120, 121, 127, 128, 129,
        130, 131, 207, 208, 209, 210, 211, 217, 218, 219, 220, 221, 227, 228,
        229, 230, 231])


In [9]:
print(max_values[30:45])
print(max_values[30:45][t.size(2)-1::t.size(2)])

tensor([207, 208, 209, 210, 211, 217, 218, 219, 220, 221, 227, 228, 229, 230,
        231])
tensor([211, 221, 231])


In [ ]:
def get_expected(test_i, t_size, batch_=0, channel_=0, heigh_=0):
    index = lambda size, b, c, h : size[2] * (size[1] * b + c) + h

    if test_i in [3, 4]:
        i_start = index(t_size, batch_, channel_, heigh_)
        i_end = index(t_size, batch_, channel_+1, heigh_)
        max_values_ = max_values[i_start:i_end]
        max_pos_ = max_pos[i_start:i_end][:,-2:]
        if test_i == 3:
            max_values_ = max_values_.view(t_size[2], 1)
        
    elif test_i in [5, 6]:
        i_start = index(t_size, batch_, channel_, heigh_)
        i_end = index(t_size, batch_, channel_+1, heigh_)
        max_values_ = max_values[i_start:i_end].max()
        max_pos_ = max_pos[i_end-1:i_end][:,-2:]
        if test_i == 5:
            max_values_ = max_values_.view(1, 1)
        else:
            max_values_ = max_values_.view(1)
    
    elif test_i in [7, 8]:
        i_start = index(t_size, batch_, channel_, heigh_)
        i_end = index(t_size, batch_+1, channel_, heigh_)
        max_values_ = max_values[i_start:i_end]
        max_pos_ = max_pos[i_start:i_end][:,1:]
        if test_i == 7:
            max_values_ = max_values_.view(t_size[1], t_size[2], 1)
        else:
            max_values_ = max_values_.view(t_size[1], t_size[2])

    elif test_i in [9, 10]:
        i_start = index(t_size, batch_, channel_, heigh_)
        i_end = index(t_size, batch_+1, channel_, heigh_)
        max_values_ = max_values[i_start+t_size[2]-1:i_end:t_size[2]]
        max_pos_ = max_pos[i_start+t_size[2]-1:i_end:t_size[2]][:,1:]
        if test_i == 9:
            max_values_ = max_values_.view(t_size[1], 1, 1)
        else:
            max_values_ = max_values_.view(t_size[1])

    elif test_i in [11, 12]:
        i_start = index(t_size, batch_, channel_, heigh_)
        i_end = index(t_size, batch_+1, channel_, heigh_)
        max_values_ = max_values[i_start+t_size[2]-1:i_end:t_size[2]].max()
        max_pos_ = max_pos[i_start+t_size[2]-1:i_end:t_size[2]][-1:][:,1:]
        if test_i == 11:
            max_values_ = max_values_.view(1, 1, 1)
        else:
            max_values_ = max_values_.unsqueeze(dim=-1)

    elif test_i in [13, 14]:
        max_values_ = max_values
        max_pos_ = max_pos
        if test_i == 13:
            max_values_ = max_values_.view(t_size[0], t_size[1], t_size[2], 1)
        else:
            max_values_ = max_values_.view(t_size[0], t_size[1], t_size[2])

    elif test_i in [15, 16]:
        max_values_ = max_values[t_size[2]-1::t_size[2]]
        max_pos_ = max_pos[t_size[2]-1::t_size[2]]
        if test_i == 15:
            max_values_ = max_values_.view(t_size[0], t_size[1], 1, 1)
        else:
            max_values_ = max_values_.view(t_size[0], t_size[1])

    elif test_i in [17, 18]:
        max_values_ = max_values[t_size[1]*t_size[2]-1::t_size[1]*t_size[2]]
        max_pos_ = max_pos[t_size[1]*t_size[2]-1::t_size[1]*t_size[2]]
        if test_i == 17:
            max_values_ = max_values_.view(t_size[0], 1, 1, 1)
        else:
            max_values_ = max_values_.view(t_size[0])

    elif test_i in [19, 20]:
        max_values_ = max_values[-1:]
        max_pos_ = max_pos[-1:]
        if test_i == 19:
            max_values_ = max_values_.view(1, 1, 1, 1)

    return (max_values_, max_pos_)


tests = [
    { # 1 : tensor 1d
        "tensor": t[*max_pos[t.size(0)][:3]],
        "expected": {
            "max_values": max_values[t.size(0)].unsqueeze(dim=0),
            "max_pos": max_pos[t.size(0)][-1].view(1, 1)
        },
        "dim": -1, # Les max de chaque ligne
        "return_pos": True,
        "keepdim": True,
    },
    { # 2 : tensor 1d
        "tensor": t[*max_pos[t.size(0)][:3]],
        "expected": {
            "max_values": max_values[t.size(0)].unsqueeze(dim=0),
            "max_pos": max_pos[t.size(0)][-1].view(1, 1)
        },
        "dim": -1, # Les max de chaque ligne
        "return_pos": True,
        "keepdim": False,
    },

    { # 3 : tensor 2d
        "tensor": t[1, 1],
        "expected": {
            "max_values": get_expected(3, t.size(), 1, 1, 0)[0],
            "max_pos": get_expected(3, t.size(), 1, 1, 0)[1]
        },
        "dim": -1, # Les max de chaque ligne
        "return_pos": True,
        "keepdim": True,
    },
    { # 4 : tensor 2d
       "tensor": t[1, 1],
        "expected": {
            "max_values": get_expected(4, t.size(), 1, 1, 0)[0],
            "max_pos": get_expected(4, t.size(), 1, 1, 0)[1]
        },
        "dim": -1, # Les max de chaque ligne
        "return_pos": True,
        "keepdim": False,
    },

    { # 5 : tensor 2d
        "tensor": t[2, 2],
        "expected": {
            "max_values": get_expected(5, t.size(), 2, 2, 0)[0],
            "max_pos": get_expected(5, t.size(), 2, 2, 0)[1]
        },
        "dim": -2, # Les max de chaque canal
        "return_pos": True,
        "keepdim": True,
    },
    { # 6 : tensor 2d
        "tensor": t[2, 2],
        "expected": {
            "max_values": get_expected(6, t.size(), 2, 2, 0)[0],
            "max_pos": get_expected(6, t.size(), 2, 2, 0)[1]
        },
        "dim": -2, # Les max de chaque canal
        "return_pos": True,
        "keepdim": False,
    },

    { # 7 : tensor 3d
        "tensor": t[1],
        "expected": {
            "max_values": get_expected(7, t.size(), 1, 0, 0)[0],
            "max_pos": get_expected(7, t.size(), 1, 0, 0)[1]
        },
        "dim": -1, # Les max de chaque ligne
        "return_pos": True,
        "keepdim": True,
    },
    { # 8 : tensor 3d
        "tensor": t[1],
        "expected": {
            "max_values": get_expected(8, t.size(), 1, 0, 0)[0],
            "max_pos": get_expected(8, t.size(), 1, 0, 0)[1]
        },
        "dim": -1, # Les max de chaque ligne
        "return_pos": True,
        "keepdim": False,
    },

    { # 9 : tensor 3d
        "tensor": t[2],
        "expected": {
            "max_values": get_expected(9, t.size(), 2, 0, 0)[0],
            "max_pos": get_expected(9, t.size(), 2, 0, 0)[1]
        },
        "dim": -2, # Les max de chaque canal
        "return_pos": True,
        "keepdim": True,
    },
    { # 10 : tensor 3d
        "tensor": t[2],
        "expected": {
            "max_values": get_expected(10, t.size(), 2, 0, 0)[0],
            "max_pos": get_expected(10, t.size(), 2, 0, 0)[1]
        },
        "dim": -2, # Les max de chaque canal
        "return_pos": True,
        "keepdim": False,
    },

    { # 11 : tensor 3d
        "tensor": t[2],
        "expected": {
            "max_values": get_expected(11, t.size(), 2, 0, 0)[0],
            "max_pos": get_expected(11, t.size(), 2, 0, 0)[1]
        },
        "dim": -3, # Les max de chaque batch
        "return_pos": True,
        "keepdim": True,
    },

    { # 12 : tensor 3d
        "tensor": t[2],
        "expected": {
            "max_values": get_expected(12, t.size(), 2, 0, 0)[0],
            "max_pos": get_expected(12, t.size(), 2, 0, 0)[1]
        },
        "dim": -3, # Les max de chaque batch
        "return_pos": True,
        "keepdim": False,
    },

    { # 13 : tensor 4d
        "tensor": t,
        "expected": {
            "max_values": get_expected(13, t.size(), 0, 0, 0)[0],
            "max_pos": get_expected(13, t.size(), 0, 0, 0)[1]
        },
        "dim": -1, # Les max de chaque ligne
        "return_pos": True,
        "keepdim": True,
    },
    { # 14 : tensor 4d
        "tensor": t,
        "expected": {
            "max_values": get_expected(14, t.size(), 0, 0, 0)[0],
            "max_pos": get_expected(14, t.size(), 0, 0, 0)[1]
        },
        "dim": -1, # Les max de chaque ligne
        "return_pos": True,
        "keepdim": False,
    },

    { # 15 : tensor 4d
        "tensor": t,
        "expected": {
            "max_values": get_expected(15, t.size(), 0, 0, 0)[0],
            "max_pos": get_expected(15, t.size(), 0, 0, 0)[1]
        },
        "dim": -2, # Les max de chaque canal
        "return_pos": True,
        "keepdim": True,
    },
    { # 16 : tensor 4d
        "tensor": t,
        "expected": {
            "max_values": get_expected(16, t.size(), 0, 0, 0)[0],
            "max_pos": get_expected(16, t.size(), 0, 0, 0)[1]
        },
        "dim": -2, # Les max de chaque canal
        "return_pos": True,
        "keepdim": False,
    },

    { # 17 : tensor 4d
        "tensor": t,
        "expected": {
            "max_values": get_expected(17, t.size(), 0, 0, 0)[0],
            "max_pos": get_expected(17, t.size(), 0, 0, 0)[1]
        },
        "dim": -3, # Les max de chaque batch
        "return_pos": True,
        "keepdim": True,
    },
    { # 18 : tensor 4d
        "tensor": t,
        "expected": {
            "max_values": get_expected(18, t.size(), 0, 0, 0)[0],
            "max_pos": get_expected(18, t.size(), 0, 0, 0)[1]
        },
        "dim": -3, # Les max de chaque batch
        "return_pos": True,
        "keepdim": False,
    },

    { # 19 : tensor 4d
        "tensor": t,
        "expected": {
            "max_values": get_expected(19, t.size(), 0, 0, 0)[0],
            "max_pos": get_expected(19, t.size(), 0, 0, 0)[1]
        },
        "dim": -4, # Les max de chaque batch
        "return_pos": True,
        "keepdim": True,
    },
    { # 20 : tensor 4d
        "tensor": t,
        "expected": {
            "max_values": get_expected(20, t.size(), 0, 0, 0)[0],
            "max_pos": get_expected(20, t.size(), 0, 0, 0)[1]
        },
        "dim": -4, # Les max de chaque batch
        "return_pos": True,
        "keepdim": False,
    },
]

for i, test in enumerate(tests, start=1):
    print(f"=== Test #{i}")
    print(f"tensor size :", test["tensor"].size())
    result = utils_pytorch.get_max_by_dim(
        test["tensor"], dim=test["dim"], return_pos=test["return_pos"], keepdim=test["keepdim"]
        )
    if test["return_pos"]:
        result_max_vector, result_max_pos = result
    else:
        result_max_vector = result
    
    print("> expected max values:", test["expected"]["max_values"].size(), test["expected"]["max_values"], sep="\n")
    print("> result_max_value :", result_max_vector.size(), result_max_vector, sep="\n")
    assert torch.equal(result_max_vector, test["expected"]["max_values"]), \
        f"Test #{i} : Expecting max_vector {test['expected']['max_values']}, using dim {test['dim']} and keepdim {test['keepdim']}"

    if test["return_pos"]:
        print("> expected pos :", test["expected"]["max_pos"], sep="\n")
        print("> result_max_pos :", result_max_pos, sep="\n")
        assert torch.equal(result_max_pos, test["expected"]["max_pos"]), \
            f"Test #{i} : Expecting max_pos {test['expected']['max_pos']}, using dim {test['dim']} and keepdim {test['keepdim']}"
    
    print()
    
print("\nOK !!!")

=== Test #1
tensor size : torch.Size([10])
> expected max values:
torch.Size([1])
tensor([10])
> result_max_value :
torch.Size([1])
tensor([10])
> expected pos :
tensor([[2]])
> result_max_pos :
tensor([[2]])

=== Test #2
tensor size : torch.Size([10])
> expected max values:
torch.Size([1])
tensor([10])
> result_max_value :
torch.Size([1])
tensor([10])
> expected pos :
tensor([[2]])
> result_max_pos :
tensor([[2]])

=== Test #3
tensor size : torch.Size([5, 10])
> expected max values:
torch.Size([5, 1])
tensor([[117],
        [118],
        [119],
        [120],
        [121]])
> result_max_value :
torch.Size([5, 1])
tensor([[117],
        [118],
        [119],
        [120],
        [121]])
> expected pos :
tensor([[0, 9],
        [1, 9],
        [2, 5],
        [3, 2],
        [4, 8]])
> result_max_pos :
tensor([[0, 9],
        [1, 9],
        [2, 5],
        [3, 2],
        [4, 8]])

=== Test #4
tensor size : torch.Size([5, 10])
> expected max values:
torch.Size([5])
tensor([117, 118

### `get_first_occurrence_indices()`

In [11]:
size = [b, c, h, w] = [2, 3, 2, 5]

n = reduce(lambda x,y : x*y, size[2:])
t_4d_r = torch.arange(0, n).view(1, 1, h, w).repeat(2, 3, 1, 1)
t_4d_r[:, :, 1, 3] = 1
print("> t_4d_r")
print(t_4d_r.size(), t_4d_r, sep="\n")

t_3d_r = t_4d_r[0]
print("> t_3d_r")
print(t_3d_r.size(), t_3d_r, sep="\n")

print("> t_4d_d")
n = reduce(lambda x,y : x*y, size)
t_4d_d = torch.arange(0, n).view(size)
print(t_4d_d.size(), t_4d_d, sep="\n")

t_3d_d = t_4d_d[0]
print("> t_3d_d")
print(t_3d_d.size(), t_3d_d, sep="\n")

> t_4d_r
torch.Size([2, 3, 2, 5])
tensor([[[[0, 1, 2, 3, 4],
          [5, 6, 7, 1, 9]],

         [[0, 1, 2, 3, 4],
          [5, 6, 7, 1, 9]],

         [[0, 1, 2, 3, 4],
          [5, 6, 7, 1, 9]]],


        [[[0, 1, 2, 3, 4],
          [5, 6, 7, 1, 9]],

         [[0, 1, 2, 3, 4],
          [5, 6, 7, 1, 9]],

         [[0, 1, 2, 3, 4],
          [5, 6, 7, 1, 9]]]])
> t_3d_r
torch.Size([3, 2, 5])
tensor([[[0, 1, 2, 3, 4],
         [5, 6, 7, 1, 9]],

        [[0, 1, 2, 3, 4],
         [5, 6, 7, 1, 9]],

        [[0, 1, 2, 3, 4],
         [5, 6, 7, 1, 9]]])
> t_4d_d
torch.Size([2, 3, 2, 5])
tensor([[[[ 0,  1,  2,  3,  4],
          [ 5,  6,  7,  8,  9]],

         [[10, 11, 12, 13, 14],
          [15, 16, 17, 18, 19]],

         [[20, 21, 22, 23, 24],
          [25, 26, 27, 28, 29]]],


        [[[30, 31, 32, 33, 34],
          [35, 36, 37, 38, 39]],

         [[40, 41, 42, 43, 44],
          [45, 46, 47, 48, 49]],

         [[50, 51, 52, 53, 54],
          [55, 56, 57, 58, 59]]]])
>

tensor 3D

In [12]:
tests = [
    { # 1
            "tensor": t_3d_r,
            "values": torch.tensor([1]),
            "expected": torch.tensor([1])
        },

        { #
            "tensor": t_3d_r,
            "values": torch.tensor([1]).view(1, 1, 1).repeat(c, 1, 1),
            "expected": torch.tensor([1]).repeat(c)
        },
        { #
            "tensor": t_3d_d,
            "values": torch.tensor([1]).view(1, 1, 1).repeat(c, 1, 1),
            "expected": torch.tensor([1])
        },
        { #
            "tensor": t_3d_r,
            "values": torch.tensor([1]).view(1, 1, 1).repeat(c, h, 1),
            "expected": torch.tensor([1]).repeat(c*h)
        },
        { #
            "tensor": t_3d_d,
            "values": torch.tensor([1]).view(1, 1, 1).repeat(c, h, 1),
            "expected": torch.tensor([1])
        },

        { #
            "tensor": t_3d_r,
            "values": torch.tensor([1, 7, 9]).view(c, 1, 1),
            "expected": torch.tensor([1, 7, 9])
        },
        { #
            "tensor": t_3d_d,
            "values": torch.tensor([7, 17, 28]).view(c, 1, 1),
            "expected": torch.tensor([7, 17, 28])
        },

        { #
            "tensor": t_3d_r,
            "values": torch.tensor([1, 6]).view(1, h, 1).repeat(c, 1, 1),
            "expected": torch.tensor([1, 6]).repeat(3)
        },
        { #
            "tensor": t_3d_d,
            "values": torch.tensor([2, 7, 12, 16, 23, 28]).view(c, h, 1),
            "expected": torch.tensor([2, 7, 12, 16, 23, 28])
        }
]

for i, test in enumerate(tests, start=1):
    #print(f"=== Test #{i}")
    #print("> tensor :", test["tensor"].size(), sep="\n")
    result = utils_pytorch.get_first_occurrence_indices(test["tensor"], test["values"])
    #print("> result :", result, sep="\n")
    check = test["tensor"][*result.T]
    #print("> check :", check, sep="\n")
    assert torch.equal(test["expected"], check), f"Test #{i} : Error {test['expected']}"
    #print()

print("OK !!!")

OK !!!


Tenseur 4D

In [13]:
tests = [
    { # 1
        "tensor": t_4d_r,
        "values": torch.tensor([1]),
        "expected": torch.tensor([1])
    },

    { #
        "tensor": t_4d_r,
        "values": torch.tensor([1]).view(1, 1, 1, 1).repeat(b, 1, 1, 1),
        "expected": torch.tensor([1]).repeat(b)
    },
    { #
        "tensor": t_4d_d,
        "values": torch.tensor([1]).view(1, 1, 1, 1).repeat(b, 1, 1, 1),
        "expected": torch.tensor([1])
    },
    { #
        "tensor": t_4d_r,
        "values": torch.tensor([1]).view(1, 1, 1, 1).repeat(b, c, 1, 1),
        "expected": torch.tensor([1]).repeat(b*c)
    },
    { #
        "tensor": t_4d_d,
        "values": torch.tensor([1]).view(1, 1, 1, 1).repeat(b, c, 1, 1),
        "expected": torch.tensor([1])
    },
    { #
        "tensor": t_4d_r,
        "values": torch.tensor([1]).view(1, 1, 1, 1).repeat(b, c, h, 1),
        "expected": torch.tensor([1]).repeat(b*c*h)
    },
    { #
        "tensor": t_4d_d,
        "values": torch.tensor([1]).view(1, 1, 1, 1).repeat(b, c, h, 1),
        "expected": torch.tensor([1])
    },

    { #
        "tensor": t_4d_r,
        "values": torch.tensor([1, 7]).view(b, 1, 1, 1),
        "expected": torch.tensor([1, 7])
    },
    { #
        "tensor": t_4d_d,
        "values": torch.tensor([7, 37]).view(b, 1, 1, 1),
        "expected": torch.tensor([7, 37])
    },

    { #
        "tensor": t_4d_r,
        "values": torch.tensor([1, 6, 9]).view(1, c, 1, 1).repeat(b, 1, 1, 1),
        "expected": torch.tensor([1, 6, 9, 1, 6, 9])
    },
    { #
        "tensor": t_4d_d,
        "values": torch.tensor([7, 12, 23, 37, 46, 55]).view(b, c, 1, 1),
        "expected": torch.tensor([7, 12, 23, 37, 46, 55])
    },
    { #
        "tensor": t_4d_d,
        "values": torch.tensor([2, 7, 12, 16, 23, 28, 31, 37, 40, 46, 54, 55]).view(b, c, h, 1),
        "expected": torch.tensor([2, 7, 12, 16, 23, 28, 31, 37, 40, 46, 54, 55])
    }
]

for i, test in enumerate(tests, start=1):
    #print(f"=== Test #{i}")
    #print("> tensor :", test["tensor"].size(), sep="\n")
    result = utils_pytorch.get_first_occurrence_indices(test["tensor"], test["values"])
    #print("> result :", result, sep="\n")
    check = test["tensor"][*result.T]
    #print("> check :", check, sep="\n")
    assert torch.equal(test["expected"], check), f"Test #{i} : Error {test['expected']}"
    #print()

print("OK !!!")

OK !!!
